# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfyuky/Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (a page).** Each row has its own `content_id`, and 32 pseudonymised clients own them. It is not a page-day and not a client.

**Time window.** This is one snapshot, counted back from the export date. The export date itself is not stored in the CSV.

| Window | Days back from export | Fields that use it |
|---|---|---|
| 90-day | 1 to 90 | `*_90d` totals, `days_with_*`, the rate columns, the impression and position tiers |
| Last 30 | 1 to 30 | `*_last_30d` |
| Previous 30 | 31 to 60 | `*_prev_30d` |
| Age fields | as of export | `content_age_days`, `days_since_last_update` and their tiers |

Days 61 to 90 only show up inside the 90-day totals. The last-30 and previous-30 windows should never add up to more than the 90-day total, and the code below checks that.

**Label (Lane 2, refresh scoring).** `is_declining_label = (trend_direction == "down")`. That means impressions in the last 30 days are more than 20% below the previous 30 days. This is an *observed proxy* for "worth a review", not proof that a page needs a refresh.

**Output.** A ranked review queue with reason codes, for a human reviewer. It is decision-support, not an automatic truth.

**My prediction before running anything:** about 30,000 rows, one per page, and a declining rate a bit over half.

In [1]:
# Setup: find the repo root (or download the starter CSV if we are on a fresh Colab), then load it
import os, sys, subprocess, urllib.request

try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    if os.path.isdir("Starter/data/raw"):
        os.chdir("Starter"); break
    if os.path.basename(os.getcwd()) == "content":   # Colab default working dir
        break
    os.chdir("..")
if not os.path.exists(CSV):
    os.makedirs("data/raw", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/yfyuky/Starter/main/" + CSV, CSV)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW t AS SELECT * FROM read_csv_auto('{CSV}')")
q = lambda sql: con.execute(sql).df()

# Claim 1: one row = one content item
print(q("""
    SELECT COUNT(*) AS rows_,
           COUNT(DISTINCT content_id) AS distinct_content_ids,
           COUNT(DISTINCT client_id)  AS clients
    FROM t""").to_string(index=False))

dupes = q("SELECT content_id, COUNT(*) c FROM t GROUP BY content_id HAVING c > 1 LIMIT 5")
print("Grain probe (rows with a repeated content_id):", len(dupes))
assert len(dupes) == 0, "grain broken: content_id is not unique"

# Claim 2: the 30-day windows sit inside the 90-day window
w = q("""
    SELECT COUNT(*) FILTER (impressions_last_30d + impressions_prev_30d >  impressions_90d) AS windows_exceed_90d,
           COUNT(*) FILTER (impressions_last_30d + impressions_prev_30d =  impressions_90d) AS windows_equal_90d,
           COUNT(*) FILTER (impressions_last_30d + impressions_prev_30d <  impressions_90d) AS days_61_to_90_have_data
    FROM t""")
print(w.to_string(index=False))
assert int(w.windows_exceed_90d[0]) == 0

# Claim 3: the label rate
print(q("""
    SELECT trend_direction, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM t GROUP BY 1 ORDER BY n DESC""").to_string(index=False))


 rows_  distinct_content_ids  clients
 30000                 30000       32
Grain probe (rows with a repeated content_id): 0


 windows_exceed_90d  windows_equal_90d  days_61_to_90_have_data
                  0               2598                    27402


trend_direction     n  pct
           down 16262 54.2
         stable  5962 19.9
             up  4388 14.6
            new  2236  7.5
           flat  1152  3.8


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

All **44 raw columns** go in exactly one bucket. `is_declining_label` is not in the raw CSV. The prep script adds it, and I rebuild it from `trend_direction` when I need it.

**Label / proxy (4).** The thing I predict, or what it is computed from. Never a feature.
`trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`. The label is a comparison of those last two.

**Feature (17).** Knowable before the moment I predict.
- keyword context: `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`
- content properties: `word_count`, `char_count`, `word_count_tier`, `char_count_tier`
- age: `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`
- earlier window: `clicks_prev_30d`, `sessions_prev_30d`. These sit in days 31 to 60, which is before the outcome window. They are close siblings of the label input `impressions_prev_30d`, so the leakage notebook has to test them.

**Context (4).** For grouping, joining, splitting and reading. The model never learns from them.
`content_id`, `client_id` (pseudonyms, and `client_id` drives the grouped split), `provider_used`, `model_used` (the dictionary marks both as not model features, and they are 71% and 19% blank).

**Excluded (19).** Each one is left out for the same reason. It is measured over a window that contains the label's outcome period (the last 30 days), so it is not knowable before I predict, and the 90-day totals cannot be split back into "safe" and "not safe" days.
- 90-day totals: `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`
- last-30 siblings: `clicks_last_30d`, `sessions_last_30d`
- rates and tiers computed from those windows: `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`

**A choice I am making on purpose.** This is the strict reading. The starter pipeline does use several of these excluded columns as features. The next notebook (`w03_feature_leakage_check`) tests that with a with-and-without comparison. If I later bring any of them back, I write it down as a deviation from this contract, with the score gap as evidence.

In [2]:
# Every raw column goes in exactly one bucket. Excluded columns carry a one-line why.
LABEL = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
FEATURE = ["search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
           "word_count", "char_count", "word_count_tier", "char_count_tier",
           "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
           "clicks_prev_30d", "sessions_prev_30d"]
CONTEXT = ["content_id", "client_id", "provider_used", "model_used"]

WHY_OVERLAP = "measured over a window that contains the label's last-30-day outcome period"
EXCLUDED = {c: "90-day total: " + WHY_OVERLAP for c in
            ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
             "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
             "days_with_impressions", "days_with_sessions"]}
EXCLUDED.update({c: "last-30 window is the outcome period itself" for c in ["clicks_last_30d", "sessions_last_30d"]})
EXCLUDED.update({c: "rate computed from 90-day totals: " + WHY_OVERLAP for c in
                 ["ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]})
EXCLUDED.update({c: "tier cut from 90-day totals: " + WHY_OVERLAP for c in ["impression_tier", "position_tier"]})

buckets = {"label": LABEL, "feature": FEATURE, "context": CONTEXT, "excluded": list(EXCLUDED)}

cols = list(q("SELECT * FROM t LIMIT 0").columns)
flat = [c for v in buckets.values() for c in v]

missing_from_contract = sorted(set(cols) - set(flat))
not_in_csv = sorted(set(flat) - set(cols))
twice = sorted({c for c in flat if flat.count(c) > 1})
print("Columns in CSV:", len(cols), "| classified:", len(flat))
print("Unclassified:", missing_from_contract, "| not in CSV:", not_in_csv, "| in two buckets:", twice)
assert not missing_from_contract and not not_in_csv and not twice

print()
print(pd.Series({k: len(v) for k, v in buckets.items()}, name="columns").to_string())
print()
print(pd.DataFrame({"why excluded": EXCLUDED}).to_string())

# The label rebuilt from its source, and a leakage guard: no label or excluded column may be a feature
df_label = q("SELECT (trend_direction = 'down')::INT AS is_declining_label FROM t")
print("\nis_declining_label rate:", round(df_label.is_declining_label.mean() * 100, 1), "%")
assert not set(FEATURE) & set(LABEL + list(EXCLUDED)), "leakage: a feature is also label/excluded"


Columns in CSV: 44 | classified: 44
Unclassified: [] | not in CSV: [] | in two buckets: []

label        4
feature     17
context      4
excluded    19

                                                                                                                        why excluded
impressions_90d                            90-day total: measured over a window that contains the label's last-30-day outcome period
clicks_90d                                 90-day total: measured over a window that contains the label's last-30-day outcome period
pageviews_90d                              90-day total: measured over a window that contains the label's last-30-day outcome period
sessions_90d                               90-day total: measured over a window that contains the label's last-30-day outcome period
users_90d                                  90-day total: measured over a window that contains the label's last-30-day outcome period
engaged_sessions_90d                       90-day


is_declining_label rate: 54.2 %


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Each claim below has a query underneath it. Where the data dictionary promised a number, the code asserts it, so a changed file makes the run fail loudly.

1. **Grain.** No repeated `content_id` (checked in section 1, re-run here with row counts per client).
2. **Counts.** 30,000 rows, 44 columns, 32 clients. Clients are very uneven in size.
3. **Missing values follow `content_type`.** The blanks are patterned, not random, so I check them per type before I ever fill anything. The plan is `has_` flags, never a blind `fillna(0)`.
4. **`avg_position = 0` means "no data".** It is not rank zero (1,205 rows), and it clusters in one content type.
5. **Rate columns are x100 percentages.** `scroll_rate` and `ai_traffic_pct` can go over 100 because they come from two measurement systems.
6. **The label follows its own rule.** `down` means below -20%, `up` means above +20%. Pages with zero previous-30 impressions can never be `down`.
7. **The label varies a lot by client.** That is the reason for a grouped split.

In [3]:
def check(name, ok):
    print(("PASS  " if ok else "CHECK ") + name)
    return ok

# 2. counts
n = q("SELECT COUNT(*) n FROM t").n[0]
check("30,000 rows", n == 30000)
check("44 columns", len(cols) == 44)
check("32 clients", q("SELECT COUNT(DISTINCT client_id) c FROM t").c[0] == 32)

per_client = q("SELECT client_id, COUNT(*) AS n FROM t GROUP BY 1 ORDER BY n DESC")
print("\nRows per client: min", per_client.n.min(), "| median", int(per_client.n.median()), "| max", per_client.n.max())
print("Top 1 / top 3 / top 5 clients hold",
      *[f"{per_client.n.head(k).sum() / n:.1%}" for k in (1, 3, 5)], "of all rows")
print("Clients with fewer than 100 rows:", int((per_client.n < 100).sum()))

# 3. missing values: overall, then by content_type
null_cols = ["search_volume", "competition", "competition_level", "cpc", "main_intent",
             "word_count", "char_count", "provider_used", "model_used", "scroll_rate", "trend_pct"]
overall = q("SELECT " + ", ".join(f"COUNT(*) FILTER ({c} IS NULL) AS {c}" for c in null_cols) + " FROM t").T
overall.columns = ["null_rows"]
overall["null_pct"] = (100 * overall.null_rows / n).round(1)
print("\nMissing values, overall:")
print(overall.to_string())
check("word_count blank in 7,699 rows", overall.null_rows["word_count"] == 7699)
check("search_volume blank in 2,468 rows", overall.null_rows["search_volume"] == 2468)
check("trend_pct blank in 3,388 rows", overall.null_rows["trend_pct"] == 3388)

by_type = q("""
    SELECT content_type, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END), 1) AS search_volume_null_pct,
           ROUND(100.0 * AVG(CASE WHEN main_intent   IS NULL THEN 1 ELSE 0 END), 1) AS main_intent_null_pct,
           ROUND(100.0 * AVG(CASE WHEN word_count    IS NULL THEN 1 ELSE 0 END), 1) AS word_count_null_pct
    FROM t GROUP BY 1 ORDER BY n DESC""")
print("\nMissing values by content_type (the pattern):")
print(by_type.to_string(index=False))

# 4. avg_position = 0 means no data
pos = q("""
    SELECT content_type, COUNT(*) FILTER (avg_position = 0) AS pos_zero,
           ROUND(100.0 * COUNT(*) FILTER (avg_position = 0) / COUNT(*), 1) AS pct
    FROM t GROUP BY 1 ORDER BY pos_zero DESC""")
print("\navg_position = 0 by content_type:")
print(pos.to_string(index=False))
check("avg_position = 0 in 1,205 rows", int(pos.pos_zero.sum()) == 1205)

# 5. rate scales
r = q("""
    SELECT MAX(ctr) AS max_ctr, MAX(engagement_rate) AS max_engagement,
           COUNT(*) FILTER (scroll_rate > 100)    AS scroll_over_100,
           COUNT(*) FILTER (ai_traffic_pct > 100) AS ai_over_100
    FROM t""")
print("\n", r.to_string(index=False))
check("ctr and engagement_rate stay within 0-100", r.max_ctr[0] <= 100 and r.max_engagement[0] <= 100)

# 6. the label follows its rule (trend_pct is rounded to 1 decimal, so allow 0.05 at the edge)
bad = q("""
    SELECT COUNT(*) FILTER (trend_direction = 'down' AND trend_pct >  -19.95) AS down_wrong,
           COUNT(*) FILTER (trend_direction = 'up'   AND trend_pct <   19.95) AS up_wrong,
           COUNT(*) FILTER (trend_direction = 'down' AND trend_pct = -20.0)   AS down_at_rounded_edge,
           COUNT(*) FILTER (impressions_prev_30d = 0 AND trend_direction = 'down') AS down_with_zero_prev
    FROM t""")
print("\n", bad.to_string(index=False))
check("label matches its threshold rule", bad.down_wrong[0] == 0 and bad.up_wrong[0] == 0)
check("no page with zero previous-30 impressions is 'down'", bad.down_with_zero_prev[0] == 0)

# 7. label rate by client (clients with at least 100 rows, so small groups do not mislead)
lab = q("""
    SELECT client_id, COUNT(*) AS n, ROUND(100.0 * AVG(CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END), 1) AS declining_pct
    FROM t GROUP BY 1 HAVING COUNT(*) >= 100 ORDER BY declining_pct""")
print("\nDeclining rate across the", len(lab), "clients with >= 100 rows:",
      lab.declining_pct.min(), "% to", lab.declining_pct.max(), "%")


PASS  30,000 rows
PASS  44 columns
PASS  32 clients



Rows per client: min 3 | median 567 | max 7008
Top 1 / top 3 / top 5 clients hold 23.4% 43.3% 56.8% of all rows
Clients with fewer than 100 rows: 8



Missing values, overall:
                   null_rows  null_pct
search_volume           2468       8.2
competition             2468       8.2
competition_level       2610       8.7
cpc                     2468       8.2
main_intent             2374       7.9
word_count              7699      25.7
char_count              7699      25.7
provider_used          21438      71.5
model_used              5733      19.1
scroll_rate              125       0.4
trend_pct               3388      11.3
PASS  word_count blank in 7,699 rows
PASS  search_volume blank in 2,468 rows
PASS  trend_pct blank in 3,388 rows



Missing values by content_type (the pattern):
      content_type     n  search_volume_null_pct  main_intent_null_pct  word_count_null_pct
   keyword article 27207                     1.4                   1.0                 28.3
    feedly article  2096                   100.0                 100.0                  0.0
comparison article   697                     0.0                   0.0                  0.0



avg_position = 0 by content_type:
      content_type  pos_zero  pct
    feedly article       730 34.8
   keyword article       475  1.7
comparison article         0  0.0
PASS  avg_position = 0 in 1,205 rows



  max_ctr  max_engagement  scroll_over_100  ai_over_100
   100.0           100.0              119           23
PASS  ctr and engagement_rate stay within 0-100

  down_wrong  up_wrong  down_at_rounded_edge  down_with_zero_prev
          0         0                     4                    0
PASS  label matches its threshold rule
PASS  no page with zero previous-30 impressions is 'down'



Declining rate across the 24 clients with >= 100 rows: 0.0 % to 93.7 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell me, with the number that backs each one:

1. **One snapshot, no history.** There is no page-by-week series, so I cannot do a time-based split or check whether a decline lasts. The label and the features both come from the same export, so any result is *observed* and *directional*.
2. **The label is a short-window proxy.** A drop of more than 20% between two 30-day windows can be seasonality or noise, not a page problem. The label rate is 54.2% overall (61.1% among pages that had any previous-30 impressions), so "declining" is the majority class here. I always print that base rate next to any score.
3. **Some pages cannot decline by construction.** 3,388 pages (11.3%) had zero previous-30 impressions. They are labelled `new` or `flat`, never `down`, so a model would be learning "not declining" for them for free. I should decide whether the review queue includes them.
4. **Clients are uneven and behave very differently.** The top client holds 23.4% of the rows, the top 3 hold 43.3%, and 8 of the 32 clients have fewer than 100 rows. The declining rate ranges from 0% to 93.7% across clients with 100+ rows. A random split would let a model memorise the client, so the honest split is grouped by `client_id`.
5. **Blank data follows content type.** All 2,096 `feedly article` rows have no keyword data, and 34.8% of them have `avg_position = 0`. Word count is blank for 28.3% of `keyword article` rows. A blind `fillna(0)` would teach the model the content type by accident.
6. **Only pages that showed up in search are here.** Every row has at least one impression and is at least 90 days old. Pages with zero impressions, and new pages, are missing, not zero. 26.6% of the rows have fewer than 100 impressions in 90 days, so single-page rates are noisy.
7. **This is a slice, not the warehouse.** 30,000 pages out of about 520,000 in the full release, and I did not find how they were sampled. I have not checked the per-client history depth or the GA4 flags from the warehouse, because those live in the gated tables. The starter CSV has no dates, so the window overlaps described in the guide cannot be checked here.
8. **Pseudonymised IDs, no text.** There are no URLs, titles, keywords or queries, so I cannot say *why* a page dropped. The output can point a reviewer to a page, but the reason has to come from the reviewer.

In [4]:
# Numbers behind each limit
lim = q("""
    SELECT COUNT(*) AS pages,
           ROUND(100.0 * AVG(CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END), 1) AS declining_pct_all,
           ROUND(100.0 * AVG(CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END) FILTER (impressions_prev_30d > 0), 1) AS declining_pct_prev_positive,
           COUNT(*) FILTER (impressions_prev_30d = 0) AS zero_prev_pages,
           ROUND(100.0 * COUNT(*) FILTER (impressions_prev_30d = 0) / COUNT(*), 1) AS zero_prev_pct,
           COUNT(*) FILTER (impressions_90d < 100) AS under_100_impr,
           ROUND(100.0 * COUNT(*) FILTER (impressions_90d < 100) / COUNT(*), 1) AS under_100_impr_pct,
           MIN(impressions_90d) AS min_impr_90d,
           MIN(content_age_days) AS min_age_days
    FROM t""")
print(lim.T.to_string(header=False))

assert lim.min_impr_90d[0] >= 1 and lim.min_age_days[0] >= 90, "population filter changed"

# Blank data by content type (limit 5) and client concentration (limit 4)
print()
print(q("""
    SELECT content_type, COUNT(*) AS pages,
           ROUND(100.0 * COUNT(*) FILTER (avg_position = 0) / COUNT(*), 1) AS position_zero_pct,
           ROUND(100.0 * COUNT(*) FILTER (word_count IS NULL) / COUNT(*), 1) AS word_count_null_pct
    FROM t GROUP BY 1 ORDER BY pages DESC""").to_string(index=False))

cnt = per_client.n
print("\nClients:", len(cnt), "| under 100 rows:", int((cnt < 100).sum()),
      "| top client share:", f"{cnt.iloc[0] / cnt.sum():.1%}")

# The starter CSV has no date column, so no time split is possible here
import re
date_like = [c for c in cols if re.search(r"(^|_)(date|month|week|day_of|report_date)(_|$)", c.lower())]
print("Date-like columns in the CSV:", date_like or "none  ->  grouped split by client_id only")


pages                        30000.0
declining_pct_all               54.2
declining_pct_prev_positive     61.1
zero_prev_pages               3388.0
zero_prev_pct                   11.3
under_100_impr                7994.0
under_100_impr_pct              26.6
min_impr_90d                     1.0
min_age_days                    90.0



      content_type  pages  position_zero_pct  word_count_null_pct
   keyword article  27207                1.7                 28.3
    feedly article   2096               34.8                  0.0
comparison article    697                0.0                  0.0

Clients: 32 | under 100 rows: 8 | top client share: 23.4%
Date-like columns in the CSV: none  ->  grouped split by client_id only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.